In [1]:
!pip install pandas deep-translator

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install googletrans==4.0.0-rc1

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

csv_path = r"C:\xampp\htdocs\TNL6323-Project\google_maps_reviews.csv"

df = pd.read_csv(csv_path)

print(f"Total reviews: {len(df)}")
df.head()

Total reviews: 200


,name,author,rating,text,date
0,poo wenqi,NaN,5,NaN,2026-06-11T23:43:58.417Z
1,Rachel Tan,NaN,5,NaN,2026-06-04T00:17:37.810Z
2,Wej Chyi Ken,NaN,5,"Too squeeze inside, sound proof also not very ...",2026-05-30T12:08:21.083Z
3,hannahh,NaN,4,the food is soo generous and the servers are s...,2026-05-27T10:37:27.805Z
4,Link Er Tan,NaN,5,NaN,2026-04-30T14:24:34.885Z


In [4]:
# Remove empty, null, or whitespace-only reviews
df = df[df["text"].notna()]          # remove NaN
df = df[df["text"].astype(str).str.strip() != ""]  # remove empty strings

# Reset index after filtering
df = df.reset_index(drop=True)

print(f"Remaining reviews: {len(df)}")
df.head()

Remaining reviews: 150


,name,author,rating,text,date
0,Wej Chyi Ken,NaN,5,"Too squeeze inside, sound proof also not very ...",2026-05-30T12:08:21.083Z
1,hannahh,NaN,4,the food is soo generous and the servers are s...,2026-05-27T10:37:27.805Z
2,Robin Lau,NaN,2,"Food is not bad, but price is slightly higher....",2026-03-29T10:58:33.491Z
3,Yeanching,NaN,5,服务好 上菜快 生蚝价格微高\n吃饭时间需要定位哦因为顾客太多了。,2026-03-28T11:12:22.638Z
4,Kai Chee Too,NaN,5,A new Japanese restaurant at malim area. Serv...,2026-03-11T15:28:59.604Z


In [5]:
reviews = df["text"].fillna("").astype(str)

print(reviews.iloc[0])

Too squeeze inside, sound proof also not very good. Difficult to chat with the persons who same table. The food is nice and service is good. Hope able upgrade the environment in future.


In [6]:
from googletrans import Translator
import pandas as pd
import time

translator = Translator()

def translate_long_text_googletrans(text, chunk_size=4000):
    if pd.isna(text) or not str(text).strip():
        return ""
    
    text = str(text).strip()
    
    # Split into chunks (sentences better, but char-based works)
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    translated_chunks = []
    
    for chunk in chunks:
        for attempt in range(3):  # retry up to 3 times
            try:
                # Detect source language (optional: force 'zh' for Chinese)
                translated = translator.translate(chunk, dest='en', src='auto')
                translated_chunks.append(translated.text)
                time.sleep(0.3)
                break
            except Exception as e:
                print(f"Attempt {attempt+1} failed: {e}")
                time.sleep(1)
        else:
            # If all retries fail, keep original chunk
            translated_chunks.append(chunk)
    
    return " ".join(translated_chunks)

# Apply to your dataframe
df["translated_text"] = df["text"].apply(translate_long_text_googletrans)

In [7]:
# ---------- SECTION: Arrange & Rename the 'number' column ----------

# Replace the 'number' column with a new sequential order (starting from 1)
df['number'] = range(1, len(df) + 1)

# 4. Display the first few rows to verify
print(df[['number', 'name', 'rating', 'translated_text']].head())

   number          name  rating  \
0       1  Wej Chyi Ken       5   
1       2       hannahh       4   
2       3     Robin Lau       2   
3       4     Yeanching       5   
4       5  Kai Chee Too       5   

                                     translated_text  
0  Too squeeze inside, sound proof also not very ...  
1  the food is soo generous and the servers are s...  
2  Food is not bad, but price is slightly higher....  
3  Good service and fast food. The price of oyste...  
4  A new Japanese restaurant at malim area.  Serv...  


In [8]:
display(
    df[
        [
            "number",
            "name",
            "rating",
            #"text",
            "translated_text"
        ]
    ].head(20)
)

,number,name,rating,translated_text
0,1,Wej Chyi Ken,5,"Too squeeze inside, sound proof also not very ..."
1,2,hannahh,4,the food is soo generous and the servers are s...
2,3,Robin Lau,2,"Food is not bad, but price is slightly higher...."
3,4,Yeanching,5,Good service and fast food. The price of oyste...
4,5,Kai Chee Too,5,A new Japanese restaurant at malim area. Serv...
5,6,Aaron Lee,5,Great services and good Japanese food here.
6,7,Lina Puah,5,Sushi San is another Japanese restaurant in th...
7,8,KHOO LI JUAN,5,Not bad~
8,9,Jun Feng,5,Sushi San is an advanced japanese food restaur...
9,10,希爾威思特,5,"Rich menu choices, good food taste and service"


In [9]:
output_path = r"C:\xampp\htdocs\TNL6323-Project\google_maps_reviews_translated.csv"

df_to_save = df[[
    "number",
    "name",
    "rating",
    #"text",
    "translated_text"
]]

df_to_save.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved to: {output_path}")

Saved to: C:\xampp\htdocs\TNL6323-Project\google_maps_reviews_translated.csv


In [10]:
for i, text in enumerate(df["translated_text"], start=1):
    print(f"{i}. {text}\n")

1. Too squeeze inside, sound proof also not very good. Difficult to chat with the persons who same table. The food is nice and service is good. Hope able upgrade the environment in future.

2. the food is soo generous and the servers are so attentive but the venue is small and kinda claustrophobic 😀

3. Food is not bad, but price is slightly higher. New place, environment is good. It was previously located in malim, i commented to them about their sushi rice bland flavour and they still serve the same flavour. Workers are all polite.

ADDED: Came since last time, waited one hour for my grilled squid and we finished all our dishes for 30 mins

4. Good service and fast food. The price of oysters is slightly high.
You need to plan your meal time because there are too many customers.

5. A new Japanese restaurant at malim area.  Serve a lot of variety of japanese cuisine. Food are delicious and well presentation. Reasonable pricing.

6. Great services and good Japanese food here.

7. Sushi